In [12]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pickle
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import RidgeClassifierCV
from sklearn.pipeline import Pipeline
import sktime.datasets
from sktime.transformations.panel.rocket import Rocket
from sktime.transformations.panel.shapelets import ContractedShapeletTransform
import time

In [4]:
dataset_df = pd.read_csv('DataSummaryShort.csv')

output_list = []

In [10]:
X_train, y_train = sktime.datasets.load_UCR_UEA_dataset('Adiac', split='train', return_X_y=True)

print(type(X_train))
print(type(y_train))
print(X_train.columns)
print(X_train.shape)
print(type(X_train['dim_0'].iloc[0]))
print(X_train['dim_0'].iloc[0])
print(y_train)

<class 'pandas.core.frame.DataFrame'>
<class 'numpy.ndarray'>
Index(['dim_0'], dtype='object')
(390, 1)
<class 'pandas.core.series.Series'>
0      1.598007
1      1.599439
2      1.570529
3      1.550474
4      1.507371
         ...   
171    1.481043
172    1.521012
173    1.564154
174    1.570855
175    1.592890
Length: 176, dtype: float64
['22' '28' '21' '15' '2' '18' '21' '36' '11' '21' '29' '26' '1' '9' '17'
 '7' '36' '25' '11' '10' '25' '14' '3' '4' '36' '4' '4' '12' '23' '23' '6'
 '22' '8' '25' '21' '34' '33' '28' '1' '24' '2' '37' '32' '37' '12' '17'
 '6' '11' '4' '29' '20' '2' '27' '17' '20' '32' '25' '30' '31' '34' '16'
 '32' '28' '23' '15' '20' '24' '11' '35' '36' '26' '12' '18' '28' '2' '14'
 '7' '28' '34' '27' '31' '31' '35' '9' '19' '36' '12' '10' '10' '7' '14'
 '15' '10' '15' '6' '4' '32' '17' '23' '33' '19' '33' '18' '22' '34' '9'
 '10' '15' '32' '30' '10' '14' '1' '27' '16' '36' '24' '22' '32' '15' '2'
 '4' '26' '27' '12' '16' '28' '24' '23' '37' '13' '4' '8' '14' '31'

In [ ]:
for dataset_index, dataset_row in dataset_df.iterrows():
    dataset_name = dataset_row['Name']
    X_train, y_train = sktime.datasets.load_UCR_UEA_dataset(dataset_name, split='train', return_X_y=True)

    print(type(X_train))
    print(type(y_train))

    start_fit = time.time()
    rocket = Rocket()  # by default, ROCKET uses 10,000 kernels
    rocket.fit(X_train)
    X_train_transform = rocket.transform(X_train)

    classifier = RidgeClassifierCV(alphas=np.logspace(-3, 3, 10), normalize=True)
    classifier.fit(X_train_transform, y_train)
    fit_time = time.time() - start_fit

    start_test = time.time()
    X_test, y_test = sktime.datasets.load_UCR_UEA_dataset(dataset_name, split="test", return_X_y=True)
    X_test_transform = rocket.transform(X_test)

    rocket_score = classifier.score(X_test_transform, y_test)
    y_rocket = classifier.predict(X_test_transform)
    test_time = time.time() - start_test

    misclassified_cases = []

    for index, (cur_test, cur_rocket) in enumerate(zip(y_test, y_rocket)):
        if cur_test != cur_rocket:
            misclassified_cases.append((index, cur_test, cur_rocket))

    output_list.append({'dataset_name':dataset_name, 'rocket_score':rocket_score, 'misclassified_cases':misclassified_cases, 'fit_time':fit_time, 'test_time':test_time})

    with open('rocket_results.pickle','wb') as op_pickle:
        pickle.dump(output_list, op_pickle)

    print("Dataset: {}, Score: {}, Number of misclassifications: {}, Total number of cases {}.".format(dataset_name, rocket_score, len(misclassified_cases), len(y_test)))

Dataset: Adiac, Score: 0.7851662404092071, Number of misclassifications: 84, Total number of cases 391.
Dataset: ArrowHead, Score: 0.8, Number of misclassifications: 35, Total number of cases 175.
Dataset: Beef, Score: 0.8333333333333334, Number of misclassifications: 5, Total number of cases 30.
Dataset: BeetleFly, Score: 0.9, Number of misclassifications: 2, Total number of cases 20.
Dataset: BirdChicken, Score: 0.9, Number of misclassifications: 2, Total number of cases 20.
Dataset: Car, Score: 0.8833333333333333, Number of misclassifications: 7, Total number of cases 60.
Dataset: CBF, Score: 1.0, Number of misclassifications: 0, Total number of cases 900.
Dataset: ChlorineConcentration, Score: 0.8184895833333333, Number of misclassifications: 697, Total number of cases 3840.
Dataset: CinCECGTorso, Score: 0.8318840579710145, Number of misclassifications: 232, Total number of cases 1380.
Dataset: Coffee, Score: 1.0, Number of misclassifications: 0, Total number of cases 28.
Dataset: 

Dataset: UWaveGestureLibraryY, Score: 0.7727526521496371, Number of misclassifications: 814, Total number of cases 3582.
Dataset: UWaveGestureLibraryZ, Score: 0.7900614182021217, Number of misclassifications: 752, Total number of cases 3582.
Dataset: Wafer, Score: 0.9977287475665152, Number of misclassifications: 14, Total number of cases 6164.
Dataset: Wine, Score: 0.7777777777777778, Number of misclassifications: 12, Total number of cases 54.
Dataset: WordSynonyms, Score: 0.7539184952978056, Number of misclassifications: 157, Total number of cases 638.


In [8]:
# How long (in minutes) to extract shapelets for.
# This is a simple lower-bound initially;
# once time is up, no further shapelets will be assessed.
time_contract_in_mins = 1

# The initial number of shapelet candidates to assess per training series.
# If all series are visited and time remains on the contract then another
# pass of the data will occur.
initial_num_shapelets_per_case = 10

output_list_shapelet = []

for dataset_index, dataset_row in dataset_df.iterrows():
    dataset_name = dataset_row['Name']
    X_train, y_train = sktime.datasets.load_UCR_UEA_dataset(dataset_name, split='train', return_X_y=True)

    start_fit = time.time()
    pipeline = Pipeline(
    [
        (
            "st",
            ContractedShapeletTransform(
                time_contract_in_mins=time_contract_in_mins,
                num_candidates_to_sample_per_case=initial_num_shapelets_per_case,
                verbose=False,
            ),
        ),
        ("rf", RandomForestClassifier(n_estimators=100)),
    ]
    )

    pipeline.fit(X_train, y_train)
    fit_time = time.time() - start_fit

    start_test = time.time()
    X_test, y_test = sktime.datasets.load_UCR_UEA_dataset(dataset_name, split='test', return_X_y=True)
    shapelet_score = pipeline.score(X_test, y_test)
    y_shapelet = pipeline.predict(X_test)
    test_time = time.time() - start_test

    misclassified_cases = []

    for index, (cur_test, cur_shapelet) in enumerate(zip(y_test, y_shapelet)):
        if cur_test != cur_shapelet:
            misclassified_cases.append((index, cur_test, cur_shapelet))

    output_list_shapelet.append({'dataset_name':dataset_name, 'shapelet_score':shapelet_score, 'misclassified_cases':misclassified_cases, 'fit_time':fit_time, 'test_time':test_time})

    with open('shapelet_results.pickle','wb') as op_pickle:
        pickle.dump(output_list_shapelet, op_pickle)

    print("Dataset: {}, Score: {}, Number of misclassifications: {}, Total number of cases {}.".format(dataset_name, shapelet_score, len(misclassified_cases), len(y_test)))

3.969597816467285
4.959986448287964
Dataset: Adiac, Score: 0.34271099744245526, Number of misclassifications: 257, Total number of cases 391.
0.4621109962463379
0.5708515644073486
0.5760045051574707
0.6582422256469727
0.7193465232849121
0.7307429313659668
0.7314074039459229
0.7920355796813965
Dataset: ArrowHead, Score: 0.7771428571428571, Number of misclassifications: 39, Total number of cases 175.
0.6048431396484375
0.9673435688018799
1.003441333770752
1.1275923252105713
1.1397068500518799
Dataset: Beef, Score: 0.4666666666666667, Number of misclassifications: 16, Total number of cases 30.
0.5919859409332275
0.6118268966674805
0.6328465938568115
0.6400132179260254
0.6685209274291992
0.7481350898742676
0.8167023658752441
Dataset: BeetleFly, Score: 0.7, Number of misclassifications: 6, Total number of cases 20.
0.8329222202301025
0.8506152629852295
0.9034144878387451
Dataset: BirdChicken, Score: 0.85, Number of misclassifications: 3, Total number of cases 20.
1.989393949508667
2.3910603

/home/hariprasad-packetai/anaconda3/envs/mpenv/lib/python3.7/site-packages/sktime/transformations/panel/shapelets.py:621: UserWarning: No valid shapelets were extracted from this dataset and calling the transform method will raise an Exception. Please re-fit the transform with other data and/or parameter options.
  "No valid shapelets were extracted from this dataset and "


RuntimeError: No shapelets were extracted in fit that exceeded the minimum information gain threshold. Please retry with other data and/or parameter settings.